# 🏥 Medical Agentic RAG

A two-collection medical RAG system powered by **OpenAI** (`gpt-4o-mini` + `text-embedding-3-small`) and orchestrated with **LangGraph**.

## Architecture

```
START → Router → Retrieve_QnA ──────────────┐
            ├── Retrieve_Device ────────────┼──→ Relevance_Checker → Augment → Generate → END
            └── Web_Search ────────────────┘         │
                     ↑                              (No)
                     └──────────────────────────────┘
```

| Component | Choice |
|---|---|
| LLM | `gpt-4o-mini` |
| Embeddings | `text-embedding-3-small` (OpenAI) |
| Vector DB | ChromaDB (persistent) |
| Web Search | Google Serper API |
| Orchestration | LangGraph |

## 1 · Install dependencies

In [ ]:
# %pip install openai chromadb langgraph langchain-community python-dotenv pandas

## 2 · Imports & configuration

In [ ]:
import os
import pandas as pd
from typing import List, Literal
from typing_extensions import TypedDict
from dotenv import load_dotenv

import chromadb
from chromadb import EmbeddingFunction, Embeddings
from openai import OpenAI
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
SERPER_API_KEY  = os.getenv("SERPER_API_KEY",  "")

LLM_MODEL    = "gpt-4o-mini"
EMBED_MODEL  = "text-embedding-3-small"
CHROMA_PATH  = "./chroma_db"
N_RESULTS    = 3
MAX_ITER     = 3

print("✅  Config loaded")

## 3 · OpenAI helpers

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

def get_llm_response(prompt: str) -> str:
    """Call gpt-4o-mini and return the text response."""
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

# Quick sanity check
print(get_llm_response("Say 'LLM ready' in exactly 3 words."))

## 4 · OpenAI Embedding Function for ChromaDB

We replace ChromaDB's default sentence-transformers with OpenAI `text-embedding-3-small`.

In [ ]:
class OpenAIEmbeddingFunction(EmbeddingFunction):
    """ChromaDB-compatible embedding function using OpenAI."""

    def __call__(self, input: List[str]) -> Embeddings:
        response = client.embeddings.create(model=EMBED_MODEL, input=input)
        return [item.embedding for item in response.data]

embed_fn = OpenAIEmbeddingFunction()
print(f"✅  Embedding function ready  (model: {EMBED_MODEL})")

## 5 · Load datasets

In [ ]:
# Dataset 1: Medical Q&A
df_qa = pd.read_csv("medical_q_n_a.csv").sample(500, random_state=0).reset_index(drop=True)
df_qa["combined_text"] = (
    "Question: " + df_qa["Question"].astype(str) + ". "
    + "Answer: "  + df_qa["Answer"].astype(str)   + ". "
    + "Type: "    + df_qa["qtype"].astype(str)    + ". "
)
print(f"Q&A dataset: {df_qa.shape}")
df_qa.head(2)

In [ ]:
# Dataset 2: Medical Device Manuals
df_dev = pd.read_csv("medical_device_manuals_dataset.csv").sample(500, random_state=0).reset_index(drop=True)
df_dev["combined_text"] = (
    "Device Name: "     + df_dev["Device_Name"].astype(str)        + ". "
    + "Model: "         + df_dev["Model_Number"].astype(str)       + ". "
    + "Manufacturer: "  + df_dev["Manufacturer"].astype(str)       + ". "
    + "Indications: "   + df_dev["Indications_for_Use"].astype(str)+ ". "
    + "Contraindications: " + df_dev["Contraindications"].fillna("None").astype(str)
)
print(f"Device dataset: {df_dev.shape}")
df_dev.head(2)

## 6 · Set up ChromaDB collections

In [ ]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection_qna = chroma_client.get_or_create_collection(
    name="medical_q_n_a",
    embedding_function=embed_fn,
)
collection_device = chroma_client.get_or_create_collection(
    name="medical_device_manual",
    embedding_function=embed_fn,
)
print("✅  Collections ready")

In [ ]:
# Upsert data (safe to re-run)
collection_qna.upsert(
    documents=df_qa["combined_text"].tolist(),
    metadatas=df_qa.to_dict(orient="records"),
    ids=df_qa.index.astype(str).tolist(),
)
collection_device.upsert(
    documents=df_dev["combined_text"].tolist(),
    metadatas=df_dev.to_dict(orient="records"),
    ids=df_dev.index.astype(str).tolist(),
)
print(f"✅  Upserted {len(df_qa)} Q&A docs and {len(df_dev)} device docs")

In [ ]:
# Quick smoke-test
test = collection_qna.query(query_texts=["What are the treatments for Kawasaki disease?"], n_results=1)
print(test["documents"][0][0][:200])

## 7 · Web Search

In [ ]:
web_search = GoogleSerperAPIWrapper()

# Quick test
print(web_search.run("Latest COVID-19 treatment guidelines 2025")[:300])

## 8 · Part A – Traditional RAG (Retriever → Augment → Generate)

A simple linear pipeline without routing or relevance-checking.

In [ ]:
class SimpleState(TypedDict):
    query: str
    context: str
    prompt: str
    response: str

def simple_retrieve(state: SimpleState) -> SimpleState:
    print("--- RETRIEVE ---")
    results = collection_qna.query(query_texts=[state["query"]], n_results=N_RESULTS)
    state["context"] = "\n".join(results["documents"][0])
    return state

def simple_augment(state: SimpleState) -> SimpleState:
    print("--- AUGMENT ---")
    state["prompt"] = f"""
Answer the following question using the context below.

Context:
{state['context']}

Question: {state['query']}

Please limit your answer to 50 words.
"""
    return state

def simple_generate(state: SimpleState) -> SimpleState:
    print("--- GENERATE ---")
    state["response"] = get_llm_response(state["prompt"])
    return state

# Build graph
simple_workflow = StateGraph(SimpleState)
simple_workflow.add_node("Retriever", simple_retrieve)
simple_workflow.add_node("Augment",   simple_augment)
simple_workflow.add_node("Generate",  simple_generate)
simple_workflow.add_edge(START,       "Retriever")
simple_workflow.add_edge("Retriever", "Augment")
simple_workflow.add_edge("Augment",   "Generate")
simple_workflow.add_edge("Generate",  END)
traditional_rag = simple_workflow.compile()

print("✅  Traditional RAG compiled")

In [ ]:
# Run it
from pprint import pprint

result = None
for step in traditional_rag.stream({"query": "What are the treatments for Kawasaki disease?"}):
    for key, val in step.items():
        result = val
pprint(result["response"])

## 9 · Part B – Agentic RAG

Adds: **LLM Router**, two retrieval collections, web-search fallback, and a **Relevance Checker** with automatic re-routing.

In [ ]:
class GraphState(TypedDict):
    query: str
    context: str
    prompt: str
    response: str
    source: str
    is_relevant: str
    iteration_count: int

In [ ]:
# ── Router ─────────────────────────────────────────────────────────────────
def router_node(state: GraphState) -> GraphState:
    print("--- ROUTER ---")
    prompt = f"""
You are a routing agent. Based on the user query, decide where to look.

Options:
- Retrieve_QnA: general medical knowledge, symptoms, diseases, treatments.
- Retrieve_Device: medical devices, device manuals, device usage.
- Web_Search: recent news, current events, tariffs, external data.

Query: "{state['query']}"

Respond ONLY with one of: Retrieve_QnA, Retrieve_Device, Web_Search
"""
    decision = get_llm_response(prompt)
    if decision not in {"Retrieve_QnA", "Retrieve_Device", "Web_Search"}:
        decision = "Web_Search"
    print(f"  → {decision}")
    state["source"] = decision
    return state

def route_decision(state: GraphState) -> str:
    return state["source"]


# ── Retrievers ──────────────────────────────────────────────────────────────
def retrieve_qna(state: GraphState) -> GraphState:
    print("--- RETRIEVE (Medical Q&A) ---")
    results = collection_qna.query(query_texts=[state["query"]], n_results=N_RESULTS)
    state["context"] = "\n".join(results["documents"][0])
    state["source"]  = "Medical Q&A Collection"
    return state

def retrieve_device(state: GraphState) -> GraphState:
    print("--- RETRIEVE (Medical Device) ---")
    results = collection_device.query(query_texts=[state["query"]], n_results=N_RESULTS)
    state["context"] = "\n".join(results["documents"][0])
    state["source"]  = "Medical Device Manual"
    return state

def web_search_node(state: GraphState) -> GraphState:
    print("--- WEB SEARCH ---")
    state["context"] = web_search.run(query=state["query"])
    state["source"]  = "Web Search"
    return state


# ── Relevance Checker ───────────────────────────────────────────────────────
def relevance_checker(state: GraphState) -> GraphState:
    print("--- RELEVANCE CHECKER ---")
    prompt = f"""
Is the context below relevant to the user query?

Context:
{state['context']}

Query: {state['query']}

Answer with only 'Yes' or 'No'.
"""
    decision = get_llm_response(prompt)
    state["is_relevant"] = "Yes" if decision.lower().startswith("y") else "No"
    print(f"  → Relevant: {state['is_relevant']}")
    return state

def relevance_decision(state: GraphState) -> str:
    count = state.get("iteration_count", 0) + 1
    state["iteration_count"] = count
    if count >= MAX_ITER:
        print(f"  → Max iterations reached – forcing answer.")
        return "Yes"
    return state["is_relevant"]


# ── Augment & Generate ──────────────────────────────────────────────────────
def augment_node(state: GraphState) -> GraphState:
    print("--- AUGMENT ---")
    state["prompt"] = f"""
Answer the following question using only the context below.

Context:
{state['context']}

Question: {state['query']}

Please limit your answer to 50 words.
"""
    return state

def generate_node(state: GraphState) -> GraphState:
    print("--- GENERATE ---")
    state["response"] = get_llm_response(state["prompt"])
    return state

In [ ]:
# Build Agentic RAG workflow
workflow = StateGraph(GraphState)

workflow.add_node("Router",           router_node)
workflow.add_node("Retrieve_QnA",     retrieve_qna)
workflow.add_node("Retrieve_Device",  retrieve_device)
workflow.add_node("Web_Search",       web_search_node)
workflow.add_node("Relevance_Checker",relevance_checker)
workflow.add_node("Augment",          augment_node)
workflow.add_node("Generate",         generate_node)

workflow.add_edge(START, "Router")
workflow.add_conditional_edges(
    "Router", route_decision,
    {"Retrieve_QnA": "Retrieve_QnA", "Retrieve_Device": "Retrieve_Device", "Web_Search": "Web_Search"},
)
workflow.add_edge("Retrieve_QnA",    "Relevance_Checker")
workflow.add_edge("Retrieve_Device", "Relevance_Checker")
workflow.add_edge("Web_Search",      "Relevance_Checker")
workflow.add_conditional_edges(
    "Relevance_Checker", relevance_decision,
    {"Yes": "Augment", "No": "Web_Search"},
)
workflow.add_edge("Augment",  "Generate")
workflow.add_edge("Generate", END)

agentic_rag = workflow.compile()

# Visualise (requires Mermaid/Graphviz)
from IPython.display import Image, display
display(Image(agentic_rag.get_graph().draw_mermaid_png()))

## 10 · Test queries

In [ ]:
def run_query(question: str) -> None:
    print("\n" + "="*60)
    print(f"QUERY: {question}")
    print("="*60)
    final = None
    for step in agentic_rag.stream({"query": question, "iteration_count": 0}):
        for key, val in step.items():
            final = val
    print(f"\n✅ Source : {final['source']}")
    print(f"   Answer : {final['response']}")

In [ ]:
# Test 1 – Medical Q&A
run_query("What are the treatments for Kawasaki disease?")

In [ ]:
# Test 2 – Medical Device
run_query("What are the usage and contraindications of a Dialysis Machine?")

In [ ]:
# Test 3 – Web Search
run_query("What are the latest COVID-19 treatment guidelines in 2025?")